# OPF fine-tune — Phase 3: train + eval (Colab GPU)

Trains the **OpenAI Privacy Filter (OPF)** token-classifier on Leizilla's committed gold
(`data/opf/gold/`) to tag structural markers of Brazilian legislation. This is Phase 3 of
the plan in [`docs/opf-finetune.md`](../docs/opf-finetune.md) (see also ADR-0012 and the
`opf-finetune` skill's `colab-and-drive.md` / `training-and-eval.md`).

**Runtime: GPU.** Set `Runtime → Change runtime type → GPU` (T4 is enough with bf16).
This notebook does **only training/eval** — data prep already happened in the repo; the
gold is the frozen, version-controlled contract.

It is safe to run on the tiny **v0** gold first: that smoke-tests the whole `opf` CLI +
Drive plumbing and gives a v0 baseline F1 *before* investing in a bigger corpus. Expect
undersampled categories (e.g. `ali_marcador` at ~7 train spans) to score F1 ≈ 0 — that is
the model correctly reporting undersampling, **not a bug**. Scale the gold afterward to
get a clean "did the data help?" delta.

### Gotchas baked in (from the skill — do not undo)
- **Mount Drive first**, before any install/download, so caching works.
- **Run OPF from the interpreter you installed it into.** Colab has no venv; install with
  `%pip` (or `uv pip install --system`) and invoke with **`sys.executable -m opf`**, never
  `uv run` (it finds the cloned repo's venv that lacks `opf`).
- **Persist the ~2.8 GB base model to Drive once**; restore from there every later run.
- **Save the checkpoint + metrics to Drive immediately** after training (runtimes are
  ephemeral).
- **Don't cargo-cult `--n-ctx 512`.** On GPU push it up — long context with no chunking is
  OPF's whole advantage; at 512 you truncate most laws mid-relatório and never see the
  dispositivo. Set as high as VRAM allows.

In [ ]:
# 0. GPU check — fail loudly if this is a CPU runtime.
import subprocess, sys
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("WARNING: nvidia-smi not found — switch to a GPU runtime "
          "(Runtime > Change runtime type > GPU). CPU works but is slow.")
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fragmentation guard

In [ ]:
# 1. Mount Drive at the very top (cache layer for base model + checkpoints).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/opf-finetune"
except Exception:
    ROOT = "/content/opf-finetune"      # non-Colab fallback (no persistence)
import os
os.makedirs(ROOT, exist_ok=True)
print("ROOT =", ROOT)

In [ ]:
# 2. Config — everything that defines a run lives here.
import datetime

# Where the gold comes from (version-controlled contract). Pin to the merged commit
# for a reproducible run; the branch is fine for iteration.
GOLD_GIT_URL = "https://github.com/franklinbaldo/leizilla.git"
GOLD_GIT_REF = "main"                 # or "claude/zealous-noether-nRabQ" until merged

CATEGORY_VERSION = "leizilla_normas_v1"   # must match data/opf/label_space.json
SEED = 13
N_CTX = 8192        # raise toward VRAM limit; v0 laws are < ~3k tokens, headroom is fine
DEVICE = "cuda"

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

LOCAL_BASE  = "/content/base/privacy_filter"
DRIVE_BASE  = f"{ROOT}/base/privacy_filter"
DATA_DIR    = f"{ROOT}/data/{CATEGORY_VERSION}"            # frozen artifact set
RUN_DIR     = f"{ROOT}/checkpoints/{CATEGORY_VERSION}/{RUN_ID}"
OUT_LOCAL   = "/content/out/best"                          # local train output
for d in (DATA_DIR, RUN_DIR, "/content/out"):
    os.makedirs(d, exist_ok=True)
print("RUN_ID =", RUN_ID, "\nDATA_DIR =", DATA_DIR, "\nRUN_DIR =", RUN_DIR)

In [ ]:
# 3. Install the OPF CLI. Colab has no venv -> install into the system interpreter,
#    then ALWAYS invoke via sys.executable (NOT `uv run`).
import os
if not os.path.exists("/content/privacy-filter"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/openai/privacy-filter",
                    "/content/privacy-filter"], check=True)
# %pip keeps the kernel's interpreter; -e exposes the `opf` CLI (redact/eval/train).
%pip install -q -e /content/privacy-filter
# Sanity: the module must import from THIS interpreter.
subprocess.run([sys.executable, "-c", "import opf, sys; print('opf OK', sys.executable)"], check=True)

In [ ]:
# Helper: run the opf CLI from the right interpreter and stream output.
def opf(*args, check=True):
    cmd = [sys.executable, "-m", "opf", *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

# Confirm the live flag surface — the skill warns flags beyond
# --validation-dataset/--label-space-json/--output-dir must be verified, not assumed.
opf("train", "--help", check=False)
opf("eval", "--help", check=False)

In [ ]:
# 4. Restore-or-fetch the base checkpoint (~2.8 GB) — download once, persist to Drive.
import os, shutil
if os.path.exists(f"{LOCAL_BASE}/config.json"):
    print("base already local this session")
elif os.path.exists(f"{DRIVE_BASE}/config.json"):
    print("restoring base from Drive…"); shutil.copytree(DRIVE_BASE, LOCAL_BASE)
else:
    print("first run: downloading base from the Hub, then persisting to Drive…")
    from huggingface_hub import snapshot_download
    snapshot_download("openai/privacy-filter", local_dir=LOCAL_BASE)
    os.makedirs(os.path.dirname(DRIVE_BASE), exist_ok=True)
    shutil.copytree(LOCAL_BASE, DRIVE_BASE)
print("base at", LOCAL_BASE)

In [ ]:
# 5. Fetch the gold from git (the frozen artifact set) into DATA_DIR.
import os, shutil, json
SRC = "/content/leizilla"
if os.path.exists(SRC):
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--depth", "1", "--branch", GOLD_GIT_REF,
                GOLD_GIT_URL, SRC], check=True)
SRC_COMMIT = subprocess.run(["git", "-C", SRC, "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
for f in ("gold/train.jsonl", "gold/val.jsonl", "gold/test.jsonl", "label_space.json"):
    shutil.copy(f"{SRC}/data/opf/{f}", f"{DATA_DIR}/{os.path.basename(f)}")
shutil.copy(f"{SRC}/data/opf/gold/manifest.json", f"{DATA_DIR}/gold_manifest.json")
print("gold @", SRC_COMMIT, "->", DATA_DIR)
print(open(f"{DATA_DIR}/gold_manifest.json", encoding="utf-8").read())

In [ ]:
# 6. Validate the gold BEFORE spending a GPU run (offset/overlap/label-space gate).
rc = subprocess.run([sys.executable, f"{SRC}/scripts/opf_annotate.py", "validate",
                     f"{DATA_DIR}/train.jsonl",
                     "--label-space", f"{DATA_DIR}/label_space.json"]).returncode
for split in ("val", "test"):
    subprocess.run([sys.executable, f"{SRC}/scripts/opf_annotate.py", "validate",
                    f"{DATA_DIR}/{split}.jsonl",
                    "--label-space", f"{DATA_DIR}/label_space.json"])
assert rc == 0, "train.jsonl failed validation — fix the gold before training"

In [ ]:
# 7. Train. Output dir gets config.json + model.safetensors + finetune_summary.json
#    + USAGE.txt. Flag names beyond the three core ones below are version-dependent —
#    if this errors, read the `opf train --help` output from the cell above and adjust
#    (e.g. the device / context-length / seed flags). The skill's
#    examples/scripts/finetuning/finetune_custom_label_demo.sh is the closest template.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
train_args = [
    "train", f"{DATA_DIR}/train.jsonl",
    "--validation-dataset", f"{DATA_DIR}/val.jsonl",
    "--label-space-json",  f"{DATA_DIR}/label_space.json",
    "--output-dir", OUT_LOCAL,
    # --- verify the following names against `opf train --help` for your opf version ---
    "--device", DEVICE,
    "--n-ctx", str(N_CTX),
    "--seed", str(SEED),
]
opf(*train_args)

In [ ]:
# 8. Persist checkpoint to Drive IMMEDIATELY (runtime can die at any time).
import shutil, os
shutil.copytree(OUT_LOCAL, RUN_DIR, dirs_exist_ok=True)
print("checkpoint saved ->", RUN_DIR)
print("contents:", os.listdir(RUN_DIR))

In [ ]:
# 9. Evaluate on the held-out PT-BR test slice. Look at span-level F1 split into
#    exact vs partial/overlap (boundary drift is OPF's failure mode on formatted text)
#    and the PER-CATEGORY breakdown with support. Capture the output to test_metrics.
import os
res = subprocess.run([sys.executable, "-m", "opf", "eval", f"{DATA_DIR}/test.jsonl",
                      "--label-space-json", f"{DATA_DIR}/label_space.json",
                      "--checkpoint", RUN_DIR],
                     capture_output=True, text=True)
print(res.stdout); print(res.stderr)
with open(f"{RUN_DIR}/test_eval.txt", "w", encoding="utf-8") as fh:
    fh.write(res.stdout + "\n" + res.stderr)

### Operating point — bias toward precision (no retrain needed)

OPF decodes with a constrained Viterbi over BIOES tags plus transition-bias parameters
(background persistence, span entry/continuation/closure, boundary handoff). Shifting
those moves the precision/recall operating point **without retraining**:

- discourage background / encourage span entry+continuation → broader spans → ↑ recall;
- the opposite → tighter, fewer spans → ↑ precision.

For legal text a false positive (mislabeling a region) usually costs more than a miss a
reviewer catches, so **bias toward precision** and keep a human-review path. Set the point
on `val`, confirm once on `test`. Exact parameter names: `opf eval --help` and the decode
config in `opf/_core/`.

In [ ]:
# 10. Qualitative check — render predicted spans inline on the test doc and eyeball
#     boundaries (one good render beats a single F1 number for spotting drift).
import json
from transformers import pipeline
clf = pipeline("token-classification", model=RUN_DIR, aggregation_strategy="simple")
test_text = json.loads(open(f"{DATA_DIR}/test.jsonl", encoding="utf-8").readline())["text"]
spans = clf(test_text[:4000])
for s in spans[:40]:
    print(f"{s['entity_group']:14s} {s['start']:5d} {test_text[s['start']:s['end']]!r}")

In [ ]:
# 11. Write the run manifest (reproducibility: ties a checkpoint to exact data + seed).
import json, datetime
run_manifest = {
    "category_version": CATEGORY_VERSION,
    "run_id": RUN_ID,
    "seed": SEED,
    "n_ctx": N_CTX,
    "device": DEVICE,
    "gold_source_commit": SRC_COMMIT,
    "gold_git_ref": GOLD_GIT_REF,
    "base_model": "openai/privacy-filter",
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open(f"{RUN_DIR}/run_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(run_manifest, fh, ensure_ascii=False, indent=2)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2))
print("\nArtifacts in", RUN_DIR, "->", os.listdir(RUN_DIR))

## Interpreting v0 — and how to scale next

**v0 is a smoke test + baseline, not a deliverable model.** With ~251 spans over 6
documents, read the result as: does the CLI/Drive chain work end-to-end, and which
categories are starved?

- `art_marcador` / `par_marcador` / `inc_marcador` have enough support (~50–100) to show a
  real signal even at v0.
- `ali_marcador` (~7 train spans), and `ementa`/`vigencia`/`revogacao` (~6 each, one per
  doc) will likely score low/zero F1. **That is the data talking, not a bug** — under ~25
  spans a small classifier can't learn a category and may even let it drag the dense ones.

**Scaling (Phase 3.5 → v1): add variety, not just volume.** Don't pour in more
Planalto-same-format laws. Target the two gaps this gold has:

1. **Laws dense in alíneas** — CLT, CDC, CPC and other coded statutes — to pull
   `ali_marcador` past ~25 and stop it underfitting every statute that uses alíneas.
2. **Emendas / leis with explicit vigência+revogação** — constitutional amendments and
   parliamentary bills have cleaner, denser `vigencia`/`revogacao` patterns than general
   laws (currently 6 each).

Keep the **Planalto-only** caveat in `manifest.json → known_limitations` exactly as is.
Real cross-fonte stratification (assembleia, casacivil, …) closes when the IA publishes
raw OCR — `opf-sample` is already wired for that and just waiting on the data.

**Sequence:** (this notebook) → v0 smoke test → scale with targeted categories → v1 train
→ compare per-category F1 deltas.